# Basic 01 - Load and Inspect
Small, linear workflow: load -> summary -> contacts/publications -> pick study/assay/run -> load data with metadata.


## 1) Imports


In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import display


## 2) Import wrapper


In [ ]:
from isa_phm import ISAWrapper


## 3) Pick ISA JSON


In [ ]:
# Set this to your dataset's ISA-PHM JSON file.
ISA_JSON = Path("path/to/your-dataset.isa-phm.json").expanduser()
DATA_ROOT = ISA_JSON.parent

print("ISA-JSON :", ISA_JSON)
print("DATA_ROOT:", DATA_ROOT)
print("Exists   :", ISA_JSON.exists())


## 4) Build wrapper


In [ ]:
wrapper = ISAWrapper(
    ISA_JSON,
    data_root=DATA_ROOT,
    strict_validation=False,
)


## 5) Summary


In [ ]:
display(wrapper.summary())


## 6) Contacts


In [ ]:
display(wrapper.contacts())


## 7) Publications


In [ ]:
display(wrapper.publications())


## 8) Pick first study


In [ ]:
study_name = wrapper.list_studies()[0].title
study = wrapper.study(study_name)
study_name


## 9) Pick first assay


In [ ]:
assay_id = study.list_assays()[0].assay_id
assay = study.assay(assay_id)
assay_id


## 10) Pick first run


In [ ]:
run_id = assay.list_runs()[0].run_id
run_id


## 11) Load one run with resolved file metadata


In [ ]:
df, meta = assay.load_dataframe_with_meta(run_id=run_id, file_type="auto")


## 12) Show load metadata


In [ ]:
display(pd.DataFrame([meta.model_dump()]))


## 13) Show first rows


In [ ]:
display(df.head(10))


## 14) Optional: AI export


In [ ]:
ctx = wrapper.ai_context(include_semantics=True, include_validation=False)
list(ctx.keys())


## 15) List AI tools


In [ ]:
tools_df = pd.DataFrame(wrapper.list_tools())
display(tools_df)


## 16) Call tool: metadata context


In [ ]:
tool_ctx = wrapper.call_tool("ai_context", {
    "include_semantics": False,
    "include_validation": False,
})
tool_ctx["ok"], sorted(tool_ctx["result"].keys())


## 17) Call tool: run data + load metadata (JSON-safe)


In [ ]:
tool_df = wrapper.call_tool("load_dataframe_with_meta", {
    "study_id": study_name,
    "assay_id": assay_id,
    "run_id": run_id,
    "file_type": "auto",
    "head_rows": 5,
})
display(pd.DataFrame([tool_df["result"]["metadata"]]))
display(pd.DataFrame(tool_df["result"]["dataframe"]["head"]))


## 18) Extended metadata tables

Use `wrapper.extensive_summary()` when you want all metadata tables at once.


In [ ]:
tables = wrapper.extensive_summary()
for name, df in tables.items():
    print(f"{name}: {df.shape}")


## 19) Clear DataFrame cache

`clear_cache()` evicts all cached DataFrames from the wrapper integrator.


In [ ]:
wrapper.clear_cache()
print("DataFrame cache cleared.")
